In [1]:
# Install required dependencies
#pip install pandas numpy xgboost matplotlib gradio groq duckdb datasets python-dotenv pyarrow
#pip install scikit-learn==1.3.2

In [2]:
# ==========================================
# CELL 1: Imports & Environment Configuration
# ==========================================
import os
import json
import numpy as np
import pandas as pd
import duckdb
import xgboost as xgb
import matplotlib.pyplot as plt
import gradio as gr
from groq import Groq
from datasets import load_dataset
from dotenv import load_dotenv

# Non-interactive backend for Matplotlib inside notebooks/Gradio
plt.switch_backend('Agg')

# Load environment secrets
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("Environment configured successfully!")

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment configured successfully!


In [3]:
# ==========================================
# CELL 2: Strict Multi-Token Intersection & Field-Aware Engine
# ==========================================
import os
import json
import duckdb
import requests
import io
import re
import numpy as np
import pandas as pd
import xgboost as xgb

_CACHED_CATALOG = None

def fetch_base_hardware_catalog() -> pd.DataFrame:
    """Fetches and prepares the remote Parquet dataset."""
    global _CACHED_CATALOG
    if _CACHED_CATALOG is not None:
        return _CACHED_CATALOG

    hf_token = os.getenv("HF_TOKEN") or globals().get("HF_TOKEN", "")
    parquet_url = "https://huggingface.co/datasets/rayyanshk/dunkai/resolve/main/hardware_dataset.parquet"
    
    if not hf_token:
        raise ValueError("Missing HF_TOKEN! Set HF_TOKEN in environment or notebook.")

    df_catalog = None

    try:
        con = duckdb.connect()
        con.execute("INSTALL httpfs; LOAD httpfs;")
        con.execute(f"CREATE SECRET IF NOT EXISTS hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
        query = f"SELECT * FROM '{parquet_url}' LIMIT 50000"
        df_catalog = con.execute(query).df()
        con.close()
    except Exception:
        pass

    if df_catalog is None or df_catalog.empty:
        headers = {"Authorization": f"Bearer {hf_token}"}
        resp = requests.get(parquet_url, headers=headers, stream=True)
        resp.raise_for_status()
        buffer = io.BytesIO(resp.content)
        df_catalog = pd.read_parquet(buffer)
        if len(df_catalog) > 50000:
            df_catalog = df_catalog.head(50000)

    df_catalog.columns = [str(c).lower().strip() for c in df_catalog.columns]
    
    # Build text fields
    df_catalog["category_str"] = df_catalog.get("category", "").fillna("").astype(str).str.lower()
    df_catalog["subcategory_str"] = df_catalog.get("subcategory", "").fillna("").astype(str).str.lower()
    df_catalog["description_str"] = df_catalog.get("description", "").fillna("").astype(str).str.lower()
    
    # Combined search document
    df_catalog["search_doc"] = (
        df_catalog["category_str"] + " " + 
        df_catalog["subcategory_str"] + " " + 
        df_catalog["description_str"]
    )

    print(f"✅ Loaded {len(df_catalog):,} baseline hardware records from Hugging Face.")
    _CACHED_CATALOG = df_catalog
    return _CACHED_CATALOG


def extract_subsystems_from_architecture_json(arch_data: dict) -> list[dict]:
    """Dynamically parses required subsystems from Architecture JSON."""
    subsystems = []
    arch_model = arch_data.get("architecture_model", {})
    
    if arch_model:
        if arch_model.get("processing_unit"):
            subsystems.append({"name": arch_model["processing_unit"], "category": "Processing", "qty": 1})
            
        for sensor in arch_model.get("sensing_modules", []):
            subsystems.append({"name": sensor, "category": "Sensors", "qty": 1})
            
        for comm in arch_model.get("communication_modules", []):
            subsystems.append({"name": comm, "category": "Communication", "qty": 1})
            
        for ui in arch_model.get("user_interface_modules", []):
            subsystems.append({"name": ui, "category": "Output", "qty": 1})
            
        for pwr in arch_model.get("power_subsystem", []):
            subsystems.append({"name": pwr, "category": "Power", "qty": 1})

    if not subsystems and "architecture_graph" in arch_data:
        nodes = arch_data["architecture_graph"].get("nodes", [])
        for node in nodes:
            data = node.get("data", {})
            subsystems.append({
                "name": data.get("label", node.get("id")),
                "category": data.get("category", "General"),
                "qty": 1
            })

    return subsystems


def train_xgboost_ranker(candidates_df: pd.DataFrame, feature_cols: list[str]) -> xgb.Booster:
    """Trains XGBoost Ranker enforcing high term-intersection scores."""
    candidates_df = candidates_df.copy()
    candidates_df["log_stock"] = np.log1p(candidates_df["stock"].fillna(0).astype(float))
    
    stock_val = candidates_df["stock"].fillna(0).astype(float)
    sim_score = candidates_df["similarity_score"].fillna(0.0).astype(float)
    
    # Strictly reward components with high semantic intersection
    relevance = np.where((sim_score >= 0.80) & (stock_val > 5), 3,
                np.where((sim_score >= 0.40) & (stock_val > 0), 2, 1))
    
    dtrain = xgb.DMatrix(
        candidates_df[feature_cols].to_numpy(),
        label=relevance,
        qid=candidates_df["numeric_qid"].to_numpy(),
        feature_names=feature_cols
    )

    params = {
        "objective": "rank:pairwise",
        "learning_rate": 0.08,
        "max_depth": 5,
        "eval_metric": "ndcg@3"
    }
    
    return xgb.train(params, dtrain, num_boost_round=40)


def calculate_dynamic_relevance(query_name: str, doc_str: str, cat_str: str) -> float:
    """Calculates semantic relevance using strict mandatory token matching."""
    tokens = [t for t in re.split(r'[\s\-_]+', query_name.lower()) if len(t) > 1]
    if not tokens:
        return 0.0

    # Categorize tokens into core requirement groups dynamically
    wireless_terms = {"wireless", "wifi", "wi-fi", "ble", "bluetooth", "rf", "802.11"}
    mcu_terms = {"mcu", "microcontroller", "processor", "soc"}
    
    has_wireless_req = any(t in wireless_terms for t in tokens)
    has_mcu_req = any(t in mcu_terms for t in tokens)

    # Rule 1: Multi-group intersection enforcement
    if has_wireless_req and has_mcu_req:
        found_wireless = any(w in doc_str for w in wireless_terms)
        found_mcu = any(m in doc_str for m in mcu_terms)
        if not (found_wireless and found_mcu):
            return 0.0  # Reject non-wireless MCUs or wireless chips without MCUs

    # Rule 2: Token coverage ratio
    matched_tokens = sum(1 for t in tokens if t in doc_str)
    coverage = matched_tokens / len(tokens)

    # Rule 3: Exact phrase and category boost
    exact_phrase_boost = 1.5 if query_name.lower() in doc_str else 1.0
    category_boost = 1.3 if any(t in cat_str for t in tokens) else 1.0

    return coverage * exact_phrase_boost * category_boost


def generate_bom_from_json_inputs(requirement_json_str: str, architecture_json_str: str):
    """Main dynamic execution pipeline."""
    try:
        req_data = json.loads(requirement_json_str) if isinstance(requirement_json_str, str) else requirement_json_str
        arch_data = json.loads(architecture_json_str) if isinstance(architecture_json_str, str) else architecture_json_str
    except Exception as exc:
        return f"### ❌ Error\nFailed to parse input JSONs: {exc}", pd.DataFrame(), None

    proj_name = req_data.get("project_name", "Hardware Design Project")
    domain = req_data.get("category", "Embedded Architecture")

    try:
        df_catalog = fetch_base_hardware_catalog()
    except Exception as e:
        return f"### ❌ Dataset Fetch Error\n{str(e)}", pd.DataFrame(), None

    subsystems = extract_subsystems_from_architecture_json(arch_data)
    if not subsystems:
        return "### ❌ Error\nNo valid subsystems found in Architecture JSON.", pd.DataFrame(), None

    PASSIVE_EXCLUSIONS = r"connector|terminal|header|plug|socket|wire-to-board|housing"

    all_candidates = []

    for sub in subsystems:
        sub_name = str(sub["name"])
        cat_name = str(sub["category"])
        qty = sub["qty"]

        working_catalog = df_catalog.copy()
        
        # Filter connectors/passives for active IC roles
        if cat_name.lower() in ["processing", "sensors", "communication", "output", "power"]:
            active_mask = ~working_catalog["search_doc"].str.contains(PASSIVE_EXCLUSIONS, na=False, regex=True)
            if active_mask.sum() > 50:
                working_catalog = working_catalog[active_mask].copy()

        # Compute dynamic relevance scores
        docs = working_catalog["search_doc"].tolist()
        cats = working_catalog["category_str"].tolist()
        
        scores = [calculate_dynamic_relevance(sub_name, d, c) for d, c in zip(docs, cats)]
        scores = np.array(scores)

        # Select top candidates with score > 0
        valid_indices = np.where(scores > 0)[0]
        if len(valid_indices) == 0:
            top_indices = scores.argsort()[::-1][:50]
        else:
            top_indices = valid_indices[scores[valid_indices].argsort()[::-1][:50]]

        sub_df = working_catalog.iloc[top_indices].copy()
        sub_df["similarity_score"] = scores[top_indices]

        sub_df["qid"] = sub_name
        sub_df["req_subsystem"] = sub_name
        sub_df["req_category"] = cat_name
        sub_df["req_qty"] = qty

        all_candidates.append(sub_df)

    candidates_df = pd.concat(all_candidates, ignore_index=True)

    candidates_df["price"] = pd.to_numeric(candidates_df["price_qty_1"], errors="coerce").fillna(1.0)
    candidates_df["stock"] = pd.to_numeric(candidates_df["stock"], errors="coerce").fillna(0)
    candidates_df["log_stock"] = np.log1p(candidates_df["stock"])

    candidates_df["mpn"] = candidates_df["mfr_part"].astype(str)
    candidates_df["manufacturer"] = candidates_df["manufacturer"].astype(str)

    qid_map = {q: idx for idx, q in enumerate(candidates_df["qid"].unique())}
    candidates_df["numeric_qid"] = candidates_df["qid"].map(qid_map)

    # ML Ranking
    feature_cols = ["similarity_score", "price", "log_stock"]
    ranker = train_xgboost_ranker(candidates_df, feature_cols)

    dmatrix = xgb.DMatrix(candidates_df[feature_cols].to_numpy(), feature_names=feature_cols)
    candidates_df["ml_rank_score"] = ranker.predict(dmatrix)

    # Strict sorting: highest term intersection first, then ML rank score
    candidates_df = candidates_df.sort_values(
        by=["numeric_qid", "similarity_score", "ml_rank_score"], 
        ascending=[True, False, False]
    )
    candidates_df["rank"] = candidates_df.groupby("numeric_qid").cumcount() + 1

    top_picks = candidates_df[candidates_df["rank"] == 1].copy()

    # Integrated System Logic (e.g., Wireless MCU absorbing standalone Wi-Fi requirement)
    mcu_pick = top_picks[top_picks["req_category"].str.lower() == "processing"]
    has_wifi_integrated = False

    if not mcu_pick.empty:
        mcu_row = mcu_pick.iloc[0]
        mcu_text = (str(mcu_row.get("mpn", "")) + " " + str(mcu_row.get("description", ""))).lower()
        if any(w in mcu_text for w in ["wifi", "wi-fi", "802.11"]):
            has_wifi_integrated = True

    final_bom_rows = []
    
    for _, row in top_picks.iterrows():
        sub_name = row["req_subsystem"]
        cat_name = row["req_category"]
        qty = int(row["req_qty"])

        if cat_name.lower() == "communication" and sub_name.lower() in ["wi-fi", "wifi"] and has_wifi_integrated:
            mcu_mpn = mcu_pick.iloc[0].get("mpn", "Wireless MCU")
            final_bom_rows.append({
                "Subsystem Module": sub_name,
                "Category": cat_name,
                "Manufacturer": mcu_pick.iloc[0].get("manufacturer", "N/A"),
                "Selected MPN": f"Integrated in MCU ({mcu_mpn})",
                "Qty": 0,
                "Unit Price ($)": 0.00,
                "Total Cost ($)": 0.00,
                "Available Stock": int(mcu_pick.iloc[0].get("stock", 0))
            })
            continue

        unit_price = round(float(row["price"]), 2)
        total_price = round(unit_price * qty, 2)

        final_bom_rows.append({
            "Subsystem Module": sub_name,
            "Category": cat_name,
            "Manufacturer": str(row["manufacturer"]),
            "Selected MPN": str(row["mpn"]),
            "Qty": qty,
            "Unit Price ($)": unit_price,
            "Total Cost ($)": total_price,
            "Available Stock": int(row["stock"])
        })

    bom_table = pd.DataFrame(final_bom_rows)

    total_cost = bom_table["Total Cost ($)"].sum()
    active_units = bom_table["Qty"].sum()

    summary_md = f"""### 🛒 Project Bill of Materials (BOM)
* **Project Name:** `{proj_name}`
* **Category / Domain:** `{domain}`
* **Selected Subsystems:** {len(bom_table)} modules ({active_units} active units)
* **Est. Total Cost:** **${total_cost:.2f} USD**
"""

    csv_path = "generated_bom.csv"
    bom_table.to_csv(csv_path, index=False)

    return summary_md, bom_table, csv_path

In [4]:
# ==========================================
# CELL 3: Interactive Gradio Application
# ==========================================
# Sample JSON Inputs
DEFAULT_REQUIREMENT_JSON = json.dumps({
  "project_name": "Weather Monitoring System",
  "domain": "Environmental IoT",
  "description": "An outdoor, solar-powered weather station that collects temperature, pressure, and humidity metrics and syncs to cloud over WiFi.",
  "operating_conditions": {
    "power_source": "Solar + LiPo Battery",
    "environment": "Outdoor Industrial"
  }
}, indent=2)

DEFAULT_ARCHITECTURE_JSON = json.dumps({
  "subsystems": [
    {
      "subsystem_name": "Main Microcontroller",
      "category": "Microcontroller",
      "required_interfaces": ["I2C", "ADC", "WiFi"],
      "max_unit_budget_usd": 5.00,
      "qty": 1
    },
    {
      "subsystem_name": "Temp & Humidity Sensor",
      "category": "Sensors",
      "required_interfaces": ["I2C"],
      "max_unit_budget_usd": 3.00,
      "qty": 1
    },
    {
      "subsystem_name": "Barometric Pressure Sensor",
      "category": "Sensors",
      "required_interfaces": ["I2C"],
      "max_unit_budget_usd": 2.50,
      "qty": 1
    },
    {
      "subsystem_name": "Solar Battery Manager",
      "category": "Power",
      "required_interfaces": ["ADC"],
      "max_unit_budget_usd": 2.00,
      "qty": 1
    }
  ]
}, indent=2)

def build_app():
    with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald")) as demo:
        gr.Markdown("# 📋 JSON-Driven Hardware BOM Generator")
        gr.Markdown("Pass your **Requirement JSON** and **Architecture JSON** to evaluate candidate hardware and produce a rank-optimized **Bill of Materials (BOM)**.")

        with gr.Row():
            with gr.Column(scale=1):
                requirement_input = gr.Code(
                    label="Project Requirement JSON",
                    language="json",
                    value=DEFAULT_REQUIREMENT_JSON,
                    lines=10
                )
                architecture_input = gr.Code(
                    label="Architecture JSON Spec",
                    language="json",
                    value=DEFAULT_ARCHITECTURE_JSON,
                    lines=12
                )
                submit_btn = gr.Button("🚀 Generate Optimized BOM", variant="primary")

            with gr.Column(scale=1.2):
                summary_output = gr.Markdown()
                bom_table_output = gr.Dataframe(
                    label="Ranked Bill of Materials (Rank #1 Selection per Subsystem)",
                    interactive=False,
                    wrap=True
                )
                csv_download_output = gr.File(label="📥 Download Exported BOM (.csv)")

        submit_btn.click(
            fn=generate_bom_from_json_inputs,
            inputs=[requirement_input, architecture_input],
            outputs=[summary_output, bom_table_output, csv_download_output]
        )

    return demo

app = build_app()

C:\Users\Admin\AppData\Local\Temp\ipykernel_13128\935904062.py:49: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="emerald")) as demo:
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\gradio\layouts\column.py:59: UserWarning: 'scale' value should be an integer. Using 1.2 will cause issues.
  warnings.warn(


In [5]:
# ==========================================
# CELL 4: Launch Application
# ==========================================
if __name__ == "__main__":
    app.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


✅ Loaded 50,000 baseline hardware records from Hugging Face.
